# 04 - 基础设施基线 (Infrastructure Baseline)

**目的**: 独立测量每个基础设施组件的原始延迟，排除应用层干扰  
**测量对象**: PostgreSQL / Redis / Qdrant / Nginx 代理开销  
**输出**: 各组件 P50/P95/Max 延迟 + 对比图表

**测量层次（排除法）**:
```
全链路延迟
  - AI Engine 内部耗时（Notebook 07）
  - LLM 调用耗时（Notebook 07）
  - Embedding 耗时（Notebook 05）
  = 基础设施开销（本 Notebook）
```

## 0. 配置参数

In [ ]:
import os
from datetime import datetime

# =============================================
# 测试参数（按需修改）
# =============================================

# 测试轮数（建议 20-50 轮获得稳定统计）
N_ITERATIONS = 30

# 服务地址
QDRANT_URL   = "http://localhost:6333"
AI_ENGINE_URL = "http://localhost:8002"   # 用于测量 Nginx 代理开销
NGINX_URL    = "http://localhost:80"      # 通过 Nginx 访问同一服务

# PostgreSQL 配置
PG_HOST     = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT     = int(os.getenv("POSTGRES_PORT", "5432"))
PG_DB       = os.getenv("POSTGRES_DB", "xiaocai")
PG_USER     = os.getenv("POSTGRES_USER", "xiaocai")
PG_PASS     = os.getenv("POSTGRES_PASSWORD", "")

# Redis
REDIS_HOST  = os.getenv("REDIS_HOST", "localhost")
REDIS_PORT  = int(os.getenv("REDIS_PORT", "6379"))

# 测试键（Redis 写入/读取）
REDIS_TEST_KEY   = "benchmark:test:key"
REDIS_TEST_VALUE = "benchmark_value_" + "x" * 100  # 100 字节 value

# Qdrant Collection（必须存在才能测试 search）
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION", "xiaocai_knowledge")
QDRANT_VECTOR_DIM = 384  # all-MiniLM-L6-v2

# 输出目录
DATA_DIR = "../data"
os.makedirs(DATA_DIR, exist_ok=True)
RESULTS_FILE = f"{DATA_DIR}/infra_baseline_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"

print(f"配置加载完成")
print(f"测试轮数: {N_ITERATIONS}")

## 1. 依赖导入

In [ ]:
import httpx
import time
import subprocess
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

print("依赖导入完成")

## 2. PostgreSQL 延迟基线

In [ ]:
def pg_exec(sql: str, timeout: int = 5) -> dict:
    """通过 docker exec 执行 PostgreSQL 命令，返回延迟"""
    cmd = f"docker exec xiaocai-postgres psql -U {PG_USER} -d {PG_DB} -t -c \"{sql}\""
    start = time.perf_counter()
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        elapsed = (time.perf_counter() - start) * 1000
        return {"success": result.returncode == 0, "latency_ms": elapsed, "output": result.stdout.strip()}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000, "output": str(e)}


print(f"PostgreSQL 延迟测试 ({N_ITERATIONS} 轮)")
print("=" * 55)

pg_scenarios = [
    ("ping",    "SELECT 1;"),
    ("session_read",   "SELECT COUNT(*) FROM sessions;"),
    ("session_write",  f"INSERT INTO sessions (session_id, data, created_at) VALUES ('bench_{random.randint(1,99999)}', '{{}}', NOW()) ON CONFLICT DO NOTHING;"),
]

pg_results = []
for i in range(N_ITERATIONS):
    for scenario_name, sql in pg_scenarios:
        r = pg_exec(sql)
        pg_results.append({"iteration": i+1, "scenario": scenario_name, "latency_ms": r["latency_ms"], "success": r["success"]})

df_pg = pd.DataFrame(pg_results)
df_pg_ok = df_pg[df_pg["success"]]

print("\nPostgreSQL 延迟统计 (ms):")
pg_stats = df_pg_ok.groupby("scenario")["latency_ms"].agg(
    P50=lambda x: np.percentile(x, 50),
    P95=lambda x: np.percentile(x, 95),
    Max="max",
    Mean="mean"
).round(1)
print(pg_stats.to_string())

if df_pg_ok.empty:
    print("注意: 所有 PostgreSQL 请求失败，检查容器运行状态")
    print("session_read/write 需要 sessions 表存在")

## 3. Redis 延迟基线

In [ ]:
def redis_exec(cmd: str, timeout: int = 3) -> dict:
    """通过 docker exec 执行 Redis 命令，返回延迟"""
    full_cmd = f"docker exec xiaocai-redis redis-cli {cmd}"
    start = time.perf_counter()
    try:
        result = subprocess.run(full_cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        elapsed = (time.perf_counter() - start) * 1000
        return {"success": result.returncode == 0, "latency_ms": elapsed, "output": result.stdout.strip()}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000, "output": str(e)}


print(f"Redis 延迟测试 ({N_ITERATIONS} 轮)")
print("=" * 55)

# 预先写入测试数据
redis_exec(f"SET {REDIS_TEST_KEY} \"{REDIS_TEST_VALUE}\"")

redis_scenarios = [
    ("ping",  f"PING"),
    ("get",   f"GET {REDIS_TEST_KEY}"),
    ("set",   f"SET {REDIS_TEST_KEY} \"{REDIS_TEST_VALUE}\""),
    ("del",   f"DEL {REDIS_TEST_KEY}"),
]

redis_results = []
for i in range(N_ITERATIONS):
    # 每轮先写一次保证 GET 有数据
    redis_exec(f"SET {REDIS_TEST_KEY} \"{REDIS_TEST_VALUE}\"")
    for scenario_name, cmd in redis_scenarios:
        r = redis_exec(cmd)
        redis_results.append({"iteration": i+1, "scenario": scenario_name, "latency_ms": r["latency_ms"], "success": r["success"]})

df_redis = pd.DataFrame(redis_results)
df_redis_ok = df_redis[df_redis["success"]]

print("\nRedis 延迟统计 (ms):")
redis_stats = df_redis_ok.groupby("scenario")["latency_ms"].agg(
    P50=lambda x: np.percentile(x, 50),
    P95=lambda x: np.percentile(x, 95),
    Max="max",
    Mean="mean"
).round(1)
print(redis_stats.to_string())

if df_redis_ok.empty:
    print("注意: 所有 Redis 请求失败，检查 xiaocai-redis 容器")

## 4. Qdrant 延迟基线

In [ ]:
def qdrant_health(url: str) -> dict:
    start = time.perf_counter()
    try:
        resp = httpx.get(f"{url}/healthz", timeout=5.0)
        return {"success": resp.status_code == 200, "latency_ms": (time.perf_counter() - start) * 1000}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000, "error": str(e)}


def qdrant_list_collections(url: str) -> dict:
    start = time.perf_counter()
    try:
        resp = httpx.get(f"{url}/collections", timeout=5.0)
        return {"success": resp.status_code == 200, "latency_ms": (time.perf_counter() - start) * 1000}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000}


def qdrant_vector_search(url: str, collection: str, dim: int, top_k: int = 5) -> dict:
    """用随机向量做搜索，测量 Qdrant 搜索延迟"""
    vector = [random.uniform(-1, 1) for _ in range(dim)]
    payload = {"vector": vector, "limit": top_k, "with_payload": True}
    start = time.perf_counter()
    try:
        resp = httpx.post(f"{url}/collections/{collection}/points/search", json=payload, timeout=10.0)
        elapsed = (time.perf_counter() - start) * 1000
        if resp.status_code == 200:
            results_count = len(resp.json().get("result", []))
            return {"success": True, "latency_ms": elapsed, "results_count": results_count}
        return {"success": False, "latency_ms": elapsed, "error": f"HTTP {resp.status_code}: {resp.text[:100]}"}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000, "error": str(e)}


print(f"Qdrant 延迟测试 ({N_ITERATIONS} 轮)")
print("=" * 55)

qdrant_results = []
for i in range(N_ITERATIONS):
    # Health check
    r_health = qdrant_health(QDRANT_URL)
    qdrant_results.append({"iteration": i+1, "scenario": "health", "latency_ms": r_health["latency_ms"], "success": r_health["success"]})

    # List collections
    r_list = qdrant_list_collections(QDRANT_URL)
    qdrant_results.append({"iteration": i+1, "scenario": "list_collections", "latency_ms": r_list["latency_ms"], "success": r_list["success"]})

    # Vector search (如果 collection 存在)
    r_search = qdrant_vector_search(QDRANT_URL, QDRANT_COLLECTION, QDRANT_VECTOR_DIM, top_k=5)
    qdrant_results.append({"iteration": i+1, "scenario": "vector_search_top5", "latency_ms": r_search["latency_ms"], "success": r_search["success"]})

df_qdrant = pd.DataFrame(qdrant_results)
df_qdrant_ok = df_qdrant[df_qdrant["success"]]

print("\nQdrant 延迟统计 (ms):")
if not df_qdrant_ok.empty:
    qdrant_stats = df_qdrant_ok.groupby("scenario")["latency_ms"].agg(
        P50=lambda x: np.percentile(x, 50),
        P95=lambda x: np.percentile(x, 95),
        Max="max",
        Mean="mean",
        SuccessRate=lambda x: len(x) / N_ITERATIONS
    ).round(2)
    print(qdrant_stats.to_string())
else:
    print("注意: Qdrant 不可达，检查容器运行状态")

# 显示 collection 是否存在
search_results = df_qdrant[df_qdrant["scenario"] == "vector_search_top5"]
if search_results["success"].sum() == 0:
    print(f"\n提示: Collection '{QDRANT_COLLECTION}' 不存在或为空")
    print("向量搜索测试失败（不影响其他指标）")

## 5. Nginx 代理开销测量

In [ ]:
def http_get_latency(url: str, path: str, timeout: float = 5.0) -> dict:
    """测量 HTTP GET 延迟"""
    full_url = f"{url}{path}"
    start = time.perf_counter()
    try:
        resp = httpx.get(full_url, timeout=timeout)
        elapsed = (time.perf_counter() - start) * 1000
        return {"success": resp.status_code < 400, "latency_ms": elapsed, "status_code": resp.status_code}
    except Exception as e:
        return {"success": False, "latency_ms": (time.perf_counter() - start) * 1000, "error": str(e)}


print(f"Nginx 代理开销测试 ({N_ITERATIONS} 轮)")
print("=" * 55)
print("方法: 对比直连 AI Engine vs 经过 Nginx 代理，差值 = Nginx 开销")

nginx_results = []
for i in range(N_ITERATIONS):
    # 直连 AI Engine /health
    r_direct = http_get_latency(AI_ENGINE_URL, "/health")
    nginx_results.append({"iteration": i+1, "scenario": "direct_ai_engine", "latency_ms": r_direct["latency_ms"], "success": r_direct["success"]})

    # 经过 Nginx → /api/v2/health (如果有路由) 或直接访问 Nginx /health
    r_nginx = http_get_latency(AI_ENGINE_URL, "/health")  # 使用相同路径对比
    r_nginx_direct = http_get_latency(NGINX_URL, "/health")  # Nginx 本身
    nginx_results.append({"iteration": i+1, "scenario": "nginx_proxy", "latency_ms": r_nginx_direct["latency_ms"], "success": r_nginx_direct["success"]})

df_nginx = pd.DataFrame(nginx_results)
df_nginx_ok = df_nginx[df_nginx["success"]]

if not df_nginx_ok.empty:
    nginx_stats = df_nginx_ok.groupby("scenario")["latency_ms"].agg(
        P50=lambda x: np.percentile(x, 50),
        P95=lambda x: np.percentile(x, 95),
        Max="max",
        Mean="mean"
    ).round(1)
    print("\nNginx 代理延迟统计 (ms):")
    print(nginx_stats.to_string())

    # 计算 Nginx 代理开销
    if "direct_ai_engine" in nginx_stats.index and "nginx_proxy" in nginx_stats.index:
        overhead_p50 = nginx_stats.loc["nginx_proxy", "P50"] - nginx_stats.loc["direct_ai_engine", "P50"]
        print(f"\nNginx 代理开销 P50: ~{overhead_p50:.1f} ms")
else:
    print("注意: Nginx 或 AI Engine 不可达")

## 6. 对比可视化

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("基础设施延迟基线 (ms)", fontsize=14)

def draw_boxplot(ax, df, group_col, val_col, title):
    if df.empty:
        ax.text(0.5, 0.5, "无数据", ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return
    groups = df[group_col].unique()
    data = [df[df[group_col] == g][val_col].values for g in groups]
    ax.boxplot(data, labels=groups, patch_artist=True)
    ax.set_title(title)
    ax.set_ylabel("延迟 (ms)")
    ax.tick_params(axis='x', rotation=15)

draw_boxplot(axes[0, 0], df_pg[df_pg["success"]], "scenario", "latency_ms", "PostgreSQL 各操作延迟")
draw_boxplot(axes[0, 1], df_redis[df_redis["success"]], "scenario", "latency_ms", "Redis 各操作延迟")
draw_boxplot(axes[1, 0], df_qdrant[df_qdrant["success"]], "scenario", "latency_ms", "Qdrant 各操作延迟")
draw_boxplot(axes[1, 1], df_nginx[df_nginx["success"]], "scenario", "latency_ms", "Nginx 代理开销")

plt.tight_layout()
chart_file = f"../data/infra_baseline_chart_{datetime.now().strftime('%Y%m%d_%H%M')}.png"
plt.savefig(chart_file, dpi=150, bbox_inches="tight")
plt.show()
print(f"图表已保存: {chart_file}")

## 7. 基础设施成本汇总表

In [ ]:
print("=" * 65)
print("基础设施延迟汇总（全链路成本分解基础）")
print("=" * 65)

summary = []

def add_summary(component, scenario, df, success_col="success"):
    d = df[df["success"]]["latency_ms"]
    if len(d) > 0:
        summary.append({
            "组件": component,
            "操作": scenario,
            "P50(ms)": round(np.percentile(d, 50), 1),
            "P95(ms)": round(np.percentile(d, 95), 1),
            "Max(ms)": round(d.max(), 1),
            "样本数": len(d),
        })

for scenario in df_pg["scenario"].unique():
    add_summary("PostgreSQL", scenario, df_pg[df_pg["scenario"] == scenario])

for scenario in df_redis["scenario"].unique():
    add_summary("Redis", scenario, df_redis[df_redis["scenario"] == scenario])

for scenario in df_qdrant["scenario"].unique():
    add_summary("Qdrant", scenario, df_qdrant[df_qdrant["scenario"] == scenario])

for scenario in df_nginx["scenario"].unique():
    add_summary("Nginx", scenario, df_nginx[df_nginx["scenario"] == scenario])

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

# 保存
all_results = pd.concat([df_pg, df_redis, df_qdrant, df_nginx], ignore_index=True)
all_results.to_csv(RESULTS_FILE, index=False)
print(f"\n详细数据已保存: {RESULTS_FILE}")